#🛠️ 0. Install MCP dependencies

In [1]:
!pip -q install fastmcp openai
import json
from fastmcp import Client


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.3/603.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.0/82.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 13.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [3]:
# Naming convention: use all lowercase and kebab-case; replace spaces, '/', and '_' with '-', keep only a-z0-9-, collapse multiple '-' and trim ends. thalhathai/MCP_SERVER_01 -> https://thalhathai-mcp-server-01.hf.space/mcp
SERVER_URL = "https://thalhathai-mcp-server-01.hf.space/mcp"  # keep the /mcp server open


In [4]:
client = Client(SERVER_URL)
await client.__aenter__()      # OPEN
await client.ping()            # sanity check; raises if unreachable
print("MCP client connected.")


MCP client connected.


In [5]:
tools_resp = await client.list_tools()
items = getattr(tools_resp, "tools", tools_resp)

print("Tools on server:")
for t in items:
    name   = getattr(t, "name", None) or getattr(t, "tool", None) or str(t)
    desc   = getattr(t, "description", "") or ""
    schema = getattr(t, "inputSchema", None) or getattr(t, "input_schema", None)
    print("•", name)
    if desc:   print("  desc:", desc)
    if schema: print("  schema:", json.dumps(schema, indent=2))



Tools on server:
• get_weather
  desc: Current weather by city name.
Returns: {temperature_c, precipitation, wind_kph, note}
  schema: {
  "additionalProperties": false,
  "properties": {
    "city": {
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "type": "object"
}
• fx_rate
  desc: Get latest FX rate between two currencies (e.g., base='MYR', target='USD').
Returns: {rate, date}
  schema: {
  "additionalProperties": false,
  "properties": {
    "base": {
      "type": "string"
    },
    "target": {
      "type": "string"
    }
  },
  "required": [
    "base",
    "target"
  ],
  "type": "object"
}
• openalex_search
  desc: Search scholarly works on OpenAlex.
Returns: {"results": [{title, first_author, year, url}], "_receipt": {...}}
  schema: {
  "additionalProperties": false,
  "properties": {
    "query": {
      "type": "string"
    },
    "limit": {
      "default": 3,
      "type": "integer"
    }
  },
  "required": [
    "query"
  ],
  "type": "object"
}
•

In [6]:
from openai import OpenAI
import json
oa = OpenAI()

SYSTEM = (
  "You are a helpful assistant. If a tool matches the task, you MUST call it, "
  "and you MUST NOT fabricate tool outputs."
)

async def chat_once_inspected(client, question, model="gpt-4o-mini", tools_for_model=None):
    # 1) discover tools if not passed in
    if tools_for_model is None:
        tools_resp = await client.list_tools()
        items = getattr(tools_resp, "tools", tools_resp)
        tools_for_model = []
        for t in items:
            name   = getattr(t, "name", None) or getattr(t, "tool", None)
            desc   = getattr(t, "description", "") or ""
            schema = getattr(t, "inputSchema", None) or getattr(t, "input_schema", None)
            if name and schema:
                tools_for_model.append({
                    "type": "function",
                    "function": {"name": name, "description": desc, "parameters": schema}
                })

    messages = [{"role":"system","content":SYSTEM},
                {"role":"user","content":question}]

    first = oa.chat.completions.create(
        model=model, messages=messages,
        tools=tools_for_model, tool_choice="auto"
    )
    msg = first.choices[0].message
    tool_calls = getattr(msg, "tool_calls", None)

    debug = {"model_first_reply": msg, "tool_calls": [], "tool_results": []}

    if tool_calls:
        # Print/record tool calls
        for tc in tool_calls:
            print(f"[MODEL REQUESTED TOOL] {tc.function.name} args={tc.function.arguments}")
            debug["tool_calls"].append({"name": tc.function.name, "args": tc.function.arguments})

            args = json.loads(tc.function.arguments or "{}")
            result = await client.call_tool(tc.function.name, args)
            payload = getattr(result, "data", result)
            print(f"[MCP RESULT] {tc.function.name} -> {payload}")
            debug["tool_results"].append({"name": tc.function.name, "result": payload})

        # Feed results back and finalize
        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type":"function",
                 "function":{"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in tool_calls
            ]
        })
        messages.extend([{
            "role":"tool", "tool_call_id": tc.id, "name": tc.function.name,
            "content": json.dumps(getattr(await client.call_tool(tc.function.name, json.loads(tc.function.arguments or "{}")), "data", {}), ensure_ascii=False)
        } for tc in tool_calls])  # if you want to re-execute; otherwise reuse payloads above

        final = oa.chat.completions.create(model=model, messages=messages)
        answer = final.choices[0].message.content
    else:
        answer = msg.content

    return answer, debug


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
# Now call the inspected chat:
answer, debug = await chat_once_inspected(
    client,  # <-- pass the client here
    "Where is CS201 held and who is the lecturer",
    model="gpt-4o-mini"
)

print("FINAL ANSWER:\n", answer)
print("\nDEBUG tool_calls:", debug["tool_calls"])
print("DEBUG tool_results:", debug["tool_results"])

[MODEL REQUESTED TOOL] campus_lookup args={"course_code":"CS201"}
[MCP RESULT] campus_lookup -> {'course_code': 'CS201', 'course': 'Data Structures', 'lecturer': 'Prof. Lim', 'room': 'Room C101', 'time': 'Tue 10:00-12:00', '_receipt': {'tool_used': True, 'server_time': '2026-02-20T10:24:56.295304+00:00', 'request_id': 'b5c02e08-dfbd-4d31-b815-bdfa462c7725'}}
FINAL ANSWER:
 CS201, which is the Data Structures course, is held in Room C101. The lecturer for this course is Prof. Lim.

DEBUG tool_calls: [{'name': 'campus_lookup', 'args': '{"course_code":"CS201"}'}]
DEBUG tool_results: [{'name': 'campus_lookup', 'result': {'course_code': 'CS201', 'course': 'Data Structures', 'lecturer': 'Prof. Lim', 'room': 'Room C101', 'time': 'Tue 10:00-12:00', '_receipt': {'tool_used': True, 'server_time': '2026-02-20T10:24:56.295304+00:00', 'request_id': 'b5c02e08-dfbd-4d31-b815-bdfa462c7725'}}}]


In [ ]:
# public holiday REMOVED ...


# fx_rate conversion Working
# get_weather Working
#  open alex retreievs papers Working
#format_citation not Working -Remove
# campus_lookup works
# timetable_conflicts works output validatoion error




#prompts


"Weather in Penang right now"

"Convert 1200 MYR to USD at today's rate; show the math"

"Where is CS201 held and who is the lecturer?"

"Check timetable conflicts"